# Notebook 5 - STD_MAE calculations



In [ ]:
import os

import pandas as pd
from joblib import Parallel, delayed

from src.clinical_combat.robust.robust_harmonization import (
    apply_script,
    calculate_mae_std,
    fit_script,
    robust_script,
)
from src.clinical_combat.robust.robust_utils import (
    get_camcan_file,
    get_metrics,
    get_site,
)


In [ ]:
MAIN_FOLDER = "RESULTS/MAE_TEST"
BASE_NAME = "harmonized"
AUGMENTATION_COPIES = 5
AUGMENTATION_SUFFIX = (
    f"_AUG_{AUGMENTATION_COPIES}" if AUGMENTATION_COPIES else ""
)
init_harmonization_method = "pairwise"
COMBAT_VARIANTS = False

metrics = get_metrics()
diseases = ["ALL"]

BASE_METHODS = ["NO", "HC", "raw"]
ROBUST_METHODS = [
    "MLP_EXAMPLE",
    "G_MAD",
    "G_ZS",
    "raw",
    "IQR",
    "SN",
    "QN",
    "MAD",
    "VS",
    "MMS",
    "ZS",
]
MLP_ONLY = ["MLP_EXAMPLE"]
ALL_METHODS = BASE_METHODS + ROBUST_METHODS

sample_sizes = None
disease_ratios = None
num_tests = None
N_JOBS = -1


## Harmonization functions


In [ ]:
def harmonize(
    train_file,
    ref_file,
    metric,
    harmonization_method,
    directory,
    robust_method,
    gt_train_file,
):
    '''Harmonize a training set and compute MAE/std.'''
    dir_path = os.path.join(directory, robust_method)

    if robust_method == "raw":
        harmonized_train_file = train_file
    else:
        model_file = fit_script(
            train_file,
            ref_file,
            metric,
            harmonization_method,
            robust_method,
            dir_path,
        )
        harmonized_train_file = apply_script(
            train_file,
            model_file,
            metric,
            harmonization_method,
            robust_method,
            dir_path,
        )

    train_df = pd.read_csv(harmonized_train_file)
    gt_train_df = pd.read_csv(gt_train_file)

    std_mae_train = calculate_mae_std(train_df, gt_train_df)
    std_mae_train["site"] = get_site(train_file)
    std_mae_train["robust_method"] = robust_method
    return std_mae_train


def analyze_site(
    train_file,
    robust_methods,
    directory,
    ref_file,
    metric,
    harmonization_method,
    gt_train_file,
):
    results = [
        harmonize(
            train_file,
            ref_file,
            metric,
            harmonization_method,
            directory,
            robust_method,
            gt_train_file,
        )
        for robust_method in robust_methods
    ]
    return pd.concat(results, ignore_index=True)


def process_analysis(
    disease,
    sample_size,
    disease_ratio,
    test_index,
    harmonization_method,
    synthetic_root,
    init_harmonization,
    metrics,
    robust_methods,
):
    size_dir = os.path.join(
        MAIN_FOLDER,
        "PROCESS",
        init_harmonization,
        harmonization_method,
        disease,
        f"{sample_size}_{int(disease_ratio * 100)}",
        f"{test_index}",
    )
    std_mae_train_path = os.path.join(
        size_dir, "std_mae_compilation_train.csv"
    )

    def load_if_exists(path):
        return pd.read_csv(path) if os.path.isfile(path) else pd.DataFrame()

    std_mae_compilation_train = load_if_exists(std_mae_train_path)

    methods_to_run = robust_methods
    if not std_mae_compilation_train.empty:
        existing = set(
            std_mae_compilation_train["robust_method"].unique()
        )
        methods_to_run = [
            method for method in robust_methods if method not in existing
        ]
        if not methods_to_run:
            print(
                f"All methods already processed for {disease} "
                f"{sample_size}_{int(disease_ratio * 100)} "
                f"test_index {test_index}."
            )
            return std_mae_train_path

    disease_dir = os.path.join(synthetic_root, disease)
    size_dir_site = os.path.join(
        disease_dir,
        f"{sample_size}_{int(disease_ratio * 100)}",
    )
    test_dir = os.path.join(size_dir_site, f"{test_index}")
    all_train_pattern = os.path.join(
        test_dir,
        f"train_{sample_size}_{int(disease_ratio * 100)}_"
        f"{test_index}_*.csv",
    )

    skip_methods = {"raw", "HC", "NO", "IQR"}
    for method in methods_to_run:
        if method not in skip_methods:
            robust_script(all_train_pattern, method)

    for metric in metrics:
        metric_dir = os.path.join(size_dir, metric)
        os.makedirs(metric_dir, exist_ok=True)

        train_filename = (
            f"train_{sample_size}_{int(disease_ratio * 100)}_"
            f"{test_index}_{metric}.csv"
        )
        gt_train_filename = os.path.join(
            test_dir,
            f"gt_train_{sample_size}_{int(disease_ratio * 100)}_"
            f"{test_index}_{metric}.csv",
        )

        train_file = os.path.join(test_dir, train_filename)
        train_df = pd.read_csv(train_file)
        train_df = train_df[
            ~train_df["bundle"].isin([
                "left_ventricle",
                "right_ventricle",
            ])
        ]
        train_df = train_df.drop(
            columns=["mean_no_cov", "metric_bundle"], errors="ignore"
        )

        train_df["site"] = f"{disease}_" + train_df["site"]
        cleaned_train_file = os.path.join(metric_dir, train_filename)
        train_df.to_csv(cleaned_train_file, index=False)

        ref_file = get_camcan_file(metric, cleaned=True)

        std_mae_train = analyze_site(
            cleaned_train_file,
            methods_to_run,
            metric_dir,
            ref_file,
            metric,
            harmonization_method,
            gt_train_filename,
        )

        std_mae_train["disease"] = disease
        std_mae_train["metric"] = metric

        std_mae_compilation_train = (
            pd.concat(
                [std_mae_compilation_train, std_mae_train],
                ignore_index=True,
            )
            .drop_duplicates()
        )

    os.makedirs(size_dir, exist_ok=True)
    std_mae_compilation_train.to_csv(std_mae_train_path, index=False)
    return std_mae_train_path


def infer_experiment_settings(base_dir, diseases):
    sample_sizes = set()
    disease_ratios = set()
    inferred_num_tests = 0

    for disease in diseases:
        disease_dir = os.path.join(base_dir, disease)
        if not os.path.isdir(disease_dir):
            continue

        for size_ratio_name in os.listdir(disease_dir):
            size_ratio_path = os.path.join(disease_dir, size_ratio_name)
            if (
                not os.path.isdir(size_ratio_path)
                or "_" not in size_ratio_name
            ):
                continue

            size_part, ratio_part = size_ratio_name.split("_", 1)
            try:
                sample_sizes.add(int(size_part))
                disease_ratios.add(int(ratio_part) / 100)
            except ValueError:
                continue

            tests = [
                int(path)
                for path in os.listdir(size_ratio_path)
                if path.isdigit()
                and os.path.isdir(os.path.join(size_ratio_path, path))
            ]
            if tests:
                inferred_num_tests = max(
                    inferred_num_tests,
                    max(tests) + 1,
                    len(tests),
                )

    if not sample_sizes or not disease_ratios or inferred_num_tests == 0:
        raise ValueError(
            f"Unable to infer parameters from {base_dir} for diseases "
            f"{diseases}."
        )

    return sorted(sample_sizes), sorted(disease_ratios), inferred_num_tests


def analyze_method(
    sample_sizes,
    disease_ratios,
    num_tests,
    robust_methods,
    diseases,
    metrics,
    harmonization_method,
    synthetic_root,
    init_harmonization,
    n_jobs=-1,
):
    if any(
        value is None
        for value in (sample_sizes, disease_ratios, num_tests)
    ):
        (
            inferred_sizes,
            inferred_ratios,
            inferred_tests,
        ) = infer_experiment_settings(synthetic_root, diseases)
        sample_sizes = inferred_sizes if sample_sizes is None else sample_sizes
        disease_ratios = (
            inferred_ratios if disease_ratios is None else disease_ratios
        )
        num_tests = inferred_tests if num_tests is None else num_tests

    tasks = [
        (
            disease,
            sample_size,
            disease_ratio,
            test_index,
            harmonization_method,
            synthetic_root,
            init_harmonization,
            metrics,
            robust_methods,
        )
        for disease in diseases
        for sample_size in sample_sizes
        for disease_ratio in disease_ratios
        for test_index in range(num_tests)
    ]

    Parallel(n_jobs=n_jobs)(
        delayed(process_analysis)(*task) for task in tasks
    )


## Run the calculations


In [ ]:

synthetic_sites_root = (
    f"DATA/processed/synthetic_sites/{init_harmonization_method}/"
    f"{BASE_NAME}{AUGMENTATION_SUFFIX}"
)
harmonization_method = "pairwise"
analyze_method(
    sample_sizes,
    disease_ratios,
    num_tests,
    ALL_METHODS,
    diseases,
    metrics,
    harmonization_method,
    synthetic_sites_root,
    init_harmonization_method,
    n_jobs=N_JOBS,
)


In [ ]:
if COMBAT_VARIANTS:
    combat_methods = ["clinic", "gam", "covbat"]
    combat_all_methods = BASE_METHODS + MLP_ONLY

    for combat_method in combat_methods:
        analyze_method(
            sample_sizes,
            disease_ratios,
            num_tests,
            combat_all_methods,
            diseases,
            metrics,
            combat_method,
            synthetic_sites_root,
            init_harmonization_method,
            n_jobs=N_JOBS,
        )
